# MITSUI model experiments in Colab

This notebook contains an experimental modeling path with generated targets,
lag/rolling/difference features, LightGBM + Random Forest + XGBoost, an
XGBoost stacking meta-model, and the Mitsui inference server.

The Colab setup configures authentication, paths, dependencies and bounded
training concurrency.


In [ ]:
!pip -q install pandas numpy polars lightgbm xgboost scikit-learn tqdm pyarrow grpcio kaggle

from pathlib import Path
import os

DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not (DATA_DIR / "train.csv").exists():
    from google.colab import userdata
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    !kaggle competitions download -c mitsui-commodity-prediction-challenge -p data/raw
    !unzip -q -o data/raw/mitsui-commodity-prediction-challenge.zip -d data/raw


# Understanding

### 📘 Understanding `target_pairs.csv` Entry

Example entry:

>
> target_0 | 1 | US_Stock_VT_adj_close
>


---

### Interpretation

| Field       | Meaning |
|-------------|---------|
| `target_0`  | The label/column name we want to predict |
| `1`         | Lag — how many days into the future to compute the return |
| `US_Stock_VT_adj_close` | The asset used for calculating the return |

---

### Formula

For a given day `d`, the target is calculated as:

>
> target_0[d] = log(val1 / val2)
>

Where:

- `val1 = Price of US_Stock_VT_adj_close at day (d + 1)`  → 🔮 future price  
- `val2 = Price of US_Stock_VT_adj_close at day (d)`      → ✅ today's price

Or, equivalently:

>
>target_0[d] = log(Price[d + 1]) - log(Price[d])
>

#  Imports

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import polars as pl
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import ProcessPoolExecutor
from functools import partial

import warnings
warnings.filterwarnings("ignore")

# Data loads

In [ ]:
ROOT = 'data/raw/'

target_pairs_df = pd.read_csv(ROOT + 'target_pairs.csv')
train_df = pd.read_csv(ROOT + 'train.csv')
train_labels_df = pd.read_csv(ROOT + "train_labels.csv")

In [ ]:
target_pairs_df.iloc[9]

means what ?

in 

```target_pairs_df.iloc[9]```

| Field       | term | mean |
|-------------|---------|---------|
| target   |                  target_9 | we want to predict |
| lag      |                         1 | days into the future |
| pair     |  FX_AUDJPY - LME_PB_Close | The asset used |

# A bit analysis and clearity

In [ ]:
lag = 1
day_id = 20  # Target day

train_data = train_df[['FX_AUDJPY','LME_PB_Close']].head(day_id)
target_data = train_labels_df['target_9'].head(day_id)

# Training features: up to (day_id - lag) → day 0 to 18
x_train = train_data.iloc[:day_id - lag]

# Training labels: shifted by lag → day 1 to 19
y_train = target_data.iloc[lag:day_id]

# To predict day 20 → use features from day 19
x_to_predict = train_data.iloc[[day_id - lag]]  # i.e., row 19

# Data prepration

In [ ]:
def generate_log_returns(data, lag):
    log_returns = pd.Series(np.nan, index=data.index)

    # Compute log returns based on the rules
    for t in range(len(data)):
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            try:
                log_returns.iloc[t] = np.log(data.iloc[t + lag + 1] / data.iloc[t + 1])
            except Exception:
                log_returns.iloc[t] = np.nan
    return log_returns


def generate_targets(column_a: pd.Series, column_b: pd.Series, lag: int) -> pd.Series:
    a_returns = generate_log_returns(column_a, lag)
    b_returns = generate_log_returns(column_b, lag)
    return a_returns - b_returns

In [ ]:
def get_data_for_day(row, index):
    """
    Return:
      target_col, x_train, x_test, y_train, y_test

    - row: one row from target_pairs_df (target, lag, pair)
    - index: integer day index (use len(train_df)-1 for latest)
    """
    target_col = row[0]          # e.g. "target_9"
    lag = int(row[1])            # ensure integer
    feature_string = row[2]      # e.g. "FX_AUDJPY-LME_PB_Close"

    # Parse features
    feature_list = [p.strip() for p in feature_string.split('-') if p.strip() != ""]
    feat_a = feature_list[0]

    # Prepare series_a and series_b (as pandas.Series)
    series_a = train_df[feat_a].iloc[:index].copy()

    if len(feature_list) > 1:
        feat_b = feature_list[1]
        series_b = train_df[feat_b].iloc[:index].copy()
    else:
        # If only one feature present, use zeros (or optionally use series_a to just return a_returns)
        series_b = pd.Series(0.0, index=train_df.index[:index])

    # Build feature_data using only existing columns
    available_cols = [c for c in feature_list if c in train_df.columns]
    feature_data = train_df[available_cols].iloc[:index].copy()
    feature_data = feature_data.fillna(method='ffill').fillna(method='bfill')

    # Compute raw y_series (may contain NaNs)
    y_series_raw = generate_targets(series_a, series_b, lag)

    # === Fill NaNs in the whole target series BEFORE slicing ===
    # (forward-fill then back-fill is typical for time-series)
    y_series = y_series_raw.fillna(method='ffill').fillna(method='bfill')

    # If the entire series is still NaN (edge case), fill with 0
    if y_series.isna().all():
        y_series = pd.Series(0.0, index=y_series_raw.index)

    # Align features and targets (account for lag)
    # x_train uses rows [0 .. index-lag-1]  (length = index - lag)
    # y_train uses rows [lag .. index-1]     (same length)
    max_idx_for_train = index - lag  # exclusive upper bound for x_train iloc
    if max_idx_for_train <= 0:
        # Not enough history to produce training rows — return empty arrays
        x_train = feature_data.iloc[0:0].copy()
        y_train = pd.Series(dtype=float)
    else:
        x_train = feature_data.iloc[:max_idx_for_train].copy()
        y_train = y_series.iloc[lag:index].copy()  # same length as x_train

    # Test feature row: features at time (index - lag)
    test_row_index = index - lag
    # Guard: if test_row_index is out of bounds use last available
    if test_row_index < 0:
        test_row_index = 0
    elif test_row_index >= len(feature_data):
        test_row_index = len(feature_data) - 1

    x_test = feature_data.iloc[[test_row_index]].copy()

    # y_test: target corresponding to prediction time (aligned to how generate_targets is defined)
    # We use the filled series to avoid NaNs
    if test_row_index < 0 or test_row_index >= len(y_series):
        # fallback to last available
        y_test = y_series.iloc[-1]
    else:
        y_test = y_series.iloc[test_row_index]

    # Ensure indices/reset for model consumption (optional)
    x_train = x_train.reset_index(drop=True)
    x_test = x_test.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True) if len(y_train) > 0 else y_train

    return target_col, x_train, x_test, y_train, y_test

def get_test_data_for_day(full_test_df , row): 
    target_col = row[0]          # "target_9"
    lag = row[1]                 # 1
    feature_string = row[2]     # "FX_AUDJPY-LME_PB_Close"
    feature_list = [p.strip() for p in feature_string.split('-')]
    # Ensure it's Pandas
    if hasattr(full_test_df, "to_pandas"):
        full_test_df = full_test_df.to_pandas()
    test_data = full_test_df[feature_list].fillna(method='ffill').fillna(method='bfill')
    return target_col , test_data

In [ ]:
# day_id = 11
# row = target_pairs_df.iloc[day_id]  
# x_train, x_test, y_train, y_test = get_data_for_day(row, day_id)
# print("="*50)
# print(x_train)
# print("="*50)
# print(x_test)
# print("="*50)
# print(y_train)
# print("="*50)
# print(y_test)

# Feature engineering

To learn theory , refer [This video on YT](https://www.youtube.com/watch?v=9QtL7m3YS9I)\
This [Tutorial](https://www.geeksforgeeks.org/data-analysis/feature-engineering-for-time-series-data-methods-and-applications/) might help .

other methods

| **#**  | **Category**                          | **Methods / Examples**                                                                                                                                                               |
| ------ | ------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| **1**  | **Lag-based features**                | - Simple lags: t−1, t−2, …<br>- Multiple-step lags (e.g., 7, 30 days)<br>- Seasonal lags (e.g., lag 12 for yearly seasonality)                                                       |
| **2**  | **Rolling / window statistics**       | - Rolling mean, median, sum, std, min, max<br>- Rolling quantiles (25%, 75%)<br>- Rolling weighted averages<br>- Rolling correlations between series<br>- Rolling skewness, kurtosis |
| **3**  | **Differences & rates of change**     | - First difference: $x_t - x_{t-1}$<br>- Percentage change: $\frac{x_t - x_{t-1}}{x_{t-1}}$<br>- Second difference<br>- Cumulative change from start/reset                           |
| **4**  | **Decomposition-based features**      | - Trend, seasonal, residual (e.g., STL)<br>- Seasonal strength / amplitude<br>- Smoothed series (moving average, exponential smoothing)                                              |
| **5**  | **Calendar / time encodings**         | - Day of week, month, quarter, year<br>- Is weekend / holiday<br>- Part of day<br>- Cyclical encoding (sin/cos)                                                                      |
| **6**  | **Fourier / frequency features**      | - Fourier series terms<br>- Spectral density / dominant frequency (FFT)                                                                                                              |
| **7**  | **Interaction & polynomial features** | - Multiplying features (e.g., price × volume)<br>- Polynomial terms (squared, cubed)<br>- Lag × lag interactions                                                                     |
| **8**  | **Aggregations over groups**          | - Group-by stats (mean, sum, std) by category<br>- Expanding window stats since start                                                                                                |
| **9**  | **Target encoding (time-aware)**      | - Mean/median of target by category **up to current time** (no leakage)                                                                                                              |
| **10** | **Anomaly / regime indicators**       | - Flags for sudden jumps/drops<br>- Regime classification (e.g., high vs. low volatility)                                                                                            |


In [ ]:
# Imputation – Forward fill, backfill, interpolate
# Outliers – Identify, dummy variable
# Transformation – Log, Box-Cox, seasonal & trend adjustment
# Encoding – One-hot, target mean, integer
# Temporal – Calendar (day, week, month), holidays, cyclical encoding
# Past features – Lag features, window features both for target and features , lag and windoww sizes
# Trend & seasonality – Time variable, changepoint, step changes, Fourier series, seasonal dummies, seasonal lags

In [ ]:
def create_lags(data, lags):
    """
    Create lag features for a pandas Series or list.

    Parameters:
    -----------
    data : pd.Series or list-like
        The original time-series data.
    lags : int or list of ints
        Lag values (e.g., 1 or [1, 2, 7]).

    Returns:
    --------
    pd.DataFrame
        Columns named 'lag_{n}' with shifted values.
    """
    s = pd.Series(data).reset_index(drop=True)
    lags = [lags] if isinstance(lags, int) else lags
    lag_df = pd.DataFrame({f'lag_{n}': s.shift(n) for n in lags})
    return lag_df

In [ ]:
def create_rolling_features(data, windows, functions=['mean']):
    """
    Create rolling window features for a pd.Series or list.

    Parameters:
    -----------
    data : pd.Series or list-like
    windows : int or list of ints
        Window sizes (e.g., 3 or [3, 7]).
    functions : str or list of str
        Aggregations to compute: 'mean', 'max', 'min', 'std', etc.

    Returns:
    --------
    pd.DataFrame
        Columns like 'roll_{func}_{w}'.
    """
    s = pd.Series(data).reset_index(drop=True)
    windows = [windows] if isinstance(windows, int) else windows
    functions = [functions] if isinstance(functions, str) else functions

    df = pd.DataFrame()
    for w in windows:
        rolled = s.rolling(window=w)
        for func in functions:
            if hasattr(rolled, func):
                df[f'roll_{func}_{w}'] = getattr(rolled, func)()
            else:
                raise ValueError(f"Unsupported function: {func}")
    return df

In [ ]:
def create_diff_features(data, lags):
    """
    Create difference-from-past features.

    Parameters:
    - data: pd.Series, list, or DataFrame column
    - lags: int or list of ints

    Returns:
    - pd.DataFrame with difference features
    """
    if isinstance(data, list):
        data = pd.Series(data)
    if isinstance(lags, int):
        lags = [lags]

    diff_df = pd.DataFrame()
    for lag in lags:
        diff_df[f'diff_{lag}'] = data.diff(lag)

    return diff_df

In [ ]:
s = pd.Series([10, 20, 30, 40, 50, 60])

print(create_lags(s, [1, 2, -1 , -2]))
print(create_rolling_features(s, windows=[2, 3], functions=['mean', 'max']))
print(create_diff_features(s, [1, 2 , -1 , -2]))

In [ ]:
def prepare_features_for_col(
    col, 
    col_name, 
    lag_values=None,
    win_values=None,
    win_methods=None,
    diff_values=None,
    is_a_target = False
):
    """
    Generate lag, rolling window, and difference features for a single column.
    
    Parameters:
    ----------
    col : list, pandas.Series, or numpy.ndarray
        The input column data.
    col_name : str
        Name of the column for naming generated features.
    lag_values : list[int]
        List of lag steps.
    win_values : list[int]
        List of window sizes for rolling features.
    win_methods : list[str]
        Methods for rolling aggregation: 'mean', 'max', 'min', 'sum', etc.
    diff_values : list[int]
        List of periods for calculating differences.
        
    Returns:
    -------
    pandas.DataFrame
        DataFrame with all generated features.
    """
    
    # Ensure input is a pandas Series
    if not isinstance(col, pd.Series):
        col = pd.Series(col)
    
    # Initialize result DataFrame
    features = pd.DataFrame(index=col.index)

    # add col as well
    if not is_a_target:
        features[f"{col_name}"] = col
    
    # --- Lag Features ---
    if lag_values:
        for lag in lag_values:
            features[f"lag_{lag}_{col_name}"] = col.shift(lag)
    
    # --- Rolling Window Features ---
    if win_values and win_methods:
        for win in win_values:
            for method in win_methods:
                if hasattr(pd.Series.rolling(col, win), method):
                    features[f"win_{method}_{win}_{col_name}"] = getattr(col.rolling(win), method)()
                else:
                    raise ValueError(f"Method '{method}' is not supported for rolling windows.")
    
    # --- Difference Features ---
    if diff_values:
        for diff in diff_values:
            features[f"diff_{diff}_{col_name}"] = col.diff(diff)
    
    return features

df_features = prepare_features_for_col(
    col=s,
    col_name="sales",
    lag_values=[1, 2,-1,-2],
    win_values=[2, 3],
    win_methods=["mean", "max"],
    diff_values=[1, 2,-1,-2]
)
df_features

In [ ]:
def prepare_features_for_df(
    df,
    lag_values=None,
    win_values=None,
    win_methods=None,
    diff_values=None
):
    to_return = pd.DataFrame()
    for col in list(df.columns):
        featdf = prepare_features_for_col(
            col=df[col],
            col_name=col,
            lag_values=lag_values,
            win_values=win_values,
            win_methods=win_methods,
            diff_values=diff_values
        )
        to_return = pd.concat([to_return,featdf],axis = 1)
    to_return = to_return.fillna(method='ffill').fillna(method='bfill')
    return to_return.fillna(method='ffill').fillna(method='bfill')

smpdf= pd.DataFrame(
    {
        'A':[3,4,5],
        'B':[4,5,6]
    }
)

smpfeatdf = prepare_features_for_df(
    smpdf,
    lag_values=[1, 2,5,7,10,15,20,-1,-2,-5,-7,-10,-20 ],
    win_values=[2, 3 , 5 , 7 , 10 , 15 , 30 ],
    win_methods=["mean", "max"],
    diff_values=[1, 2, 5 , 7 , 10 , 15 , 20 ,-1,-2, -5,-7,-10,-15,-20 ]
)
smpfeatdf

In [ ]:
# scaling
scaler = StandardScaler()
def scale(df):
    scaled_array = scaler.fit_transform(df)
    scaled_df = pd.DataFrame(scaled_array, columns=df.columns)
    return scaled_df

print(scale(smpfeatdf.fillna(0)).head(2))

# logging
def log_transform_df(df):
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df_log = df.copy()
    df_log[numeric_cols] = np.log1p(df_log[numeric_cols])  # log1p = log(x+1)
    return df_log

print(log_transform_df(smpfeatdf).head(2))

# Model engineering

In [ ]:
def mean_squared_error(preds, trues):
    preds = np.array(preds)
    trues = np.array(trues)
    return np.mean((preds - trues) ** 2)

def train_and_get_result(x_train, y_train, x_test):
    # Define base learners
    base_learners = [
        ('lgbm', LGBMRegressor(n_estimators=50, learning_rate=0.1, random_state=42, verbosity=-1)),
        ('rf', RandomForestRegressor(n_estimators=50, random_state=42)),
        ('xgb', XGBRegressor(n_estimators=50, random_state=42))
    ]
    # Step 1: Train base models and get predictions on training data
    meta_features_train = []
    meta_features_test = []
    base_models = []
    for name, model in base_learners:
        model.fit(x_train, y_train)
        train_pred = model.predict(x_train)
        test_pred = model.predict(x_test)
        meta_features_train.append(train_pred.reshape(-1, 1))  # For stacking
        meta_features_test.append(test_pred.reshape(-1, 1))    # For prediction
        base_models.append(model)
    # Stack base learners' predictions as features for meta-model
    meta_x_train = np.hstack(meta_features_train)
    meta_x_test = np.hstack(meta_features_test)
    # Step 2: Train meta-model (final estimator)
    meta_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
    meta_model.fit(meta_x_train, y_train)
    # Step 3: Predict using the meta-model
    final_pred = meta_model.predict(meta_x_test)[0]
    # Optional: Save the meta-model
    # joblib.dump(meta_model, "final_stacked_xgb_model.pkl")
    return final_pred, {"meta_model" : meta_model , "base_models" : base_models }

def predict_on_models(model_dict, test):
    """
    Use saved base models to generate predictions and feed them into the meta-model.
    """
    base_models = model_dict["base_models"]
    meta_model = model_dict["meta_model"]
    # Get base models' predictions on the test input
    meta_features_test = []
    for model in base_models:
        pred = model.predict(test)
        meta_features_test.append(pred.reshape(-1, 1))  # Ensure 2D shape
    # Stack all base predictions into one feature set
    meta_x_test = np.hstack(meta_features_test)
    # Final prediction using the meta-model
    final_pred = meta_model.predict(meta_x_test)
    return final_pred

def safe_log1p(X):
    X = np.array(X, dtype=float)
    X = np.where(X <= -1, np.nan, X)  # avoid log on invalid values
    return np.log1p(X)

In [ ]:
day_id = 12
row = target_pairs_df.iloc[day_id]  
target_col , x_train, x_test, y_train, y_test = get_data_for_day(row, day_id)
x_train_feat = prepare_features_for_df(
    x_train,
    lag_values=[1, 2,-1,-2],
    win_values=[2, 3],
    win_methods=["mean", "max"],
    diff_values=[1, 2,-1,-2]
)
x_test_feat = prepare_features_for_df(
    x_test,
    lag_values=[1, 2,-1,-2],
    win_values=[2, 3],
    win_methods=["mean", "max"],
    diff_values=[1, 2,-1,-2]
)
pipeline = Pipeline([
    ('log', FunctionTransformer(safe_log1p, validate=False)),
    ('replace_inf', FunctionTransformer(lambda X: np.where(np.isinf(X), np.nan, X), validate=False)),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

x_train_feat_scaled = pipeline.fit_transform(x_train_feat)
x_test_feat_scaled = pipeline.transform(x_test_feat)

value , model_dict = train_and_get_result(x_train_feat_scaled, y_train, x_test_feat_scaled)
value

In [ ]:
TARGET_MODEL_POOL = {}
TARGET_MODEL_PIPELINE_POOL = {}

def replace_inf_func(X):
    return np.where(np.isinf(X), np.nan, X)

def train_one_target(row, day_id):
    target_col, x_train, x_test, y_train, y_test = get_data_for_day(row, day_id)

    # Prepare features
    x_train_feat = prepare_features_for_df(
        x_train,
        lag_values=[1, 2, -1, -2],
        win_values=[2, 3],
        win_methods=["mean", "max"],
        diff_values=[1, 2, -1, -2]
    )
    x_test_feat = prepare_features_for_df(
        x_test,
        lag_values=[1, 2, -1, -2],
        win_values=[2, 3],
        win_methods=["mean", "max"],
        diff_values=[1, 2, -1, -2]
    )

    # Build pipeline without lambdas
    pipeline = Pipeline([
        ('log', FunctionTransformer(safe_log1p, validate=False)),
        ('replace_inf', FunctionTransformer(replace_inf_func, validate=False)),
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    # Scale data
    x_train_feat_sc = pipeline.fit_transform(x_train_feat)
    x_test_feat_sc = pipeline.transform(x_test_feat)

    # Train model
    value, model_dict = train_and_get_result(x_train_feat_sc, y_train, x_test_feat_sc)

    return target_col, pipeline, model_dict

# Determine day_id
if len(train_df) >= 2:
    day_id = train_df.index[-2]
else:
    raise ValueError("Not enough data for training")

rows = [target_pairs_df.iloc[d] for d in range(len(target_pairs_df))]

# Use partial to avoid lambda
with ThreadPoolExecutor(max_workers=min(8, len(rows))) as executor:
    for target_col, pipeline, model_dict in executor.map(partial(train_one_target, day_id=day_id), rows):
        TARGET_MODEL_PIPELINE_POOL[target_col] = pipeline
        TARGET_MODEL_POOL[target_col] = model_dict

In [ ]:
def safe_fillna(df):
    # Replace NaN medians with 0 as fallback
    medians = df.median()
    medians = medians.fillna(0)
    return df.fillna(medians)

def _predict_one_target(row, full_test_df):
    target_col, test_df = get_test_data_for_day(full_test_df, row)

    # Fill before feature creation
    test_df = safe_fillna(test_df)

    # Create features
    test_df_feat = prepare_features_for_df(
        test_df,
        lag_values=[1, 2, -1, -2],
        win_values=[2, 3],
        win_methods=["mean", "max"],
        diff_values=[1, 2, -1, -2]
    )

    pipeline = TARGET_MODEL_PIPELINE_POOL[target_col]
    test_df_feat_sc = pipeline.transform(test_df_feat)

    # Predict
    md = TARGET_MODEL_POOL[target_col]
    predictions = predict_on_models(md, test_df_feat_sc)

    return target_col, predictions

predict_invoke_count = 0

def predict_on_test(full_test_df):
    global predict_invoke_count
    print(f"predict invoked {predict_invoke_count}")
    rows = [target_pairs_df.iloc[t] for t in range(len(target_pairs_df))]
    preds_dict = {}

    with ThreadPoolExecutor() as executor:
        for target_col, predictions in executor.map(lambda row: _predict_one_target(row, full_test_df), rows):
            preds_dict[target_col] = predictions

    preds_df = pd.DataFrame(preds_dict)
    # preds_df.to_parquet("submission.parquet", index=False)
    predict_invoke_count += 1
    return preds_df

In [ ]:
preds_for_test_set = predict_on_test(pd.read_csv(ROOT + 'test.csv'))
preds_for_test_set.sample(3)

# submission

In [ ]:
import sys
sys.path.append(str(Path('data/raw').resolve()))

In [ ]:
def predict(
    test: pl.DataFrame,
    label_lags_1_batch: pl.DataFrame,
    label_lags_2_batch: pl.DataFrame,
    label_lags_3_batch: pl.DataFrame,
    label_lags_4_batch: pl.DataFrame,
) -> pd.DataFrame:
    # Convert Polars to Pandas
    test = test.to_pandas()
    label_lags_1_batch = label_lags_1_batch.to_pandas()
    label_lags_2_batch = label_lags_2_batch.to_pandas()
    label_lags_3_batch = label_lags_3_batch.to_pandas()
    label_lags_4_batch = label_lags_4_batch.to_pandas()
    return predict_on_test(test)

In [ ]:
from kaggle_evaluation.mitsui_inference_server import MitsuiInferenceServer

inference_server = MitsuiInferenceServer(predict)

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    inference_server.serve()
else:
    # Local testing — only works if test.csv is present
    inference_server.run_local_gateway((str(Path('data/raw').resolve()),))